# Step 1 (v2): Joint image + text analysis with Claude Vision

**Run this notebook on YOUR computer** (needs internet + an Anthropic API key).

## What changed from v1
The first version asked Claude to look at the painting alone and guess generic labels (color tone, composition, etc.). Those labels were not tied to any real research question.

This version sends Claude **both the image AND the student's emotion-writing text together**, and asks 4 labels that are each grounded in a specific theory/paper and map directly onto a testable hypothesis:

| Label | What it captures | Theoretical grounding |
|---|---|---|
| `depicted_imagery` | What the drawing actually depicts | Baseline for matching against text |
| `imagery_intensity_vs_text` | Does the drawing amplify, contradict, match, or soften the emotional intensity of the written text? | E-THER's verbal-visual incongruence framework (minimizing / contradiction / no-incongruence) |
| `imagery_specificity` | Does the drawing generically depict the metaphor, or precisely render its specific details? | Concreteness vs. specificity distinction in semantics research |
| `depicted_body_parts` | Which body parts/organs appear in the drawing (list) | For item-by-item matching against body parts mentioned in text |

Because labels 2 and 3 require comparing the image AGAINST the text, **this version sends both to Claude in the same request** (not image alone).

## Output
`output/image_labels.csv` with columns:
`student_id, depicted_imagery, imagery_intensity_vs_text, imagery_specificity, depicted_body_parts, raw_response`

## 0. Install dependencies (run once)

In [ ]:
%pip install anthropic pandas

## 1. Configuration — use an environment variable

Set `ANTHROPIC_API_KEY` in your shell before launching Jupyter. Never paste or commit API keys into a notebook.

In [ ]:
import os
from pathlib import Path

# ============== CONFIG ==============
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
if not API_KEY:
    raise RuntimeError("Set ANTHROPIC_API_KEY before running this notebook.")
def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        real_file = candidate / "Dataset" / "art_data.xlsx"
        example_file = candidate / "Dataset" / "example" / "art_data.xlsx"
        if real_file.is_file() or example_file.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find Dataset/art_data.xlsx or Dataset/example/art_data.xlsx. Start Jupyter from the project root."
    )

REPO_ROOT = find_repo_root()
DATASET_DIR = REPO_ROOT / "Dataset"
REAL_DATA_FILE = DATASET_DIR / "art_data.xlsx"
EXAMPLE_DATA_FILE = DATASET_DIR / "example" / "art_data.xlsx"
DATA_FILE = REAL_DATA_FILE if REAL_DATA_FILE.is_file() else EXAMPLE_DATA_FILE
USING_EXAMPLE_DATA = DATA_FILE == EXAMPLE_DATA_FILE
OUTPUT_DIR = REPO_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "image_labels.csv"
IMAGE_COLUMN = "processed_painting_path"   # change to "painting_path" if you prefer the originals
MODEL = "claude-sonnet-4-6"
MAX_RETRIES = 3
SLEEP_BETWEEN_CALLS = 0.3   # seconds, gentle rate limiting
# =====================================



## 2. Imports and client setup

In [ ]:
import csv
import json
import base64
import time

import anthropic
import pandas as pd

client = anthropic.Anthropic(api_key=API_KEY)

print("Client ready.")

## 3. The prompt (image + text sent together)

In [ ]:
def build_prompt(student_text):
    text_block = student_text if student_text and student_text != "na" else "(the student did not write anything for this step)"
    return f"""You are analyzing a drawing made by an adolescent (age 13-17) during an art therapy session, together with what they wrote about their feelings before drawing.

STUDENT'S WRITTEN TEXT (original Chinese, describing a body sensation and often a metaphor using "\u50cf" / "like"):
\"{text_block}\"

Look at the attached drawing and compare it against this text. Respond with ONLY a JSON object (no other text, no markdown fences) with exactly these fields:

{{
  "depicted_imagery": "1-5 words describing what the drawing actually depicts (e.g. 'two masks', 'tree with roots', 'abstract shapes', 'bird', 'brain with food')",
  "imagery_intensity_vs_text": "one of: amplifies, contradicts, matches, softens, no_text_to_compare - compare the EMOTIONAL INTENSITY/VALENCE depicted in the drawing against what the text says. 'amplifies' = drawing shows stronger/darker emotion than the text implies. 'contradicts' = drawing shows opposite emotional valence from the text (e.g. text is negative but drawing looks bright/positive, or vice versa). 'matches' = drawing's emotional tone is consistent with the text. 'softens' = drawing shows weaker/gentler emotion than the text implies. Use 'no_text_to_compare' only if the text field above says the student wrote nothing.",
  "imagery_specificity": "one of: generic, partial, precise, no_text_to_compare - if the text contains a metaphor (e.g. 'like a bird', 'like a stone'), does the drawing depict that metaphor only generically (any generic version of that object) or does it precisely render specific details implied by the text (e.g. text says 'heavy like a stone' and the drawing shows something visibly weighted down, sinking, or crushing)? Use 'no_text_to_compare' if there is no metaphor in the text to compare against.",
  "depicted_body_parts": "comma-separated list of any body parts, organs, or face/figure elements visible in the drawing (e.g. 'heart, eyes' or 'none' if no body element is depicted)"
}}

Respond with ONLY the JSON object, nothing else."""

print("Prompt builder defined.")

## 4. Helper functions

In [ ]:
def encode_image(path):
    with open(path, "rb") as f:
        data = f.read()
    ext = path.lower().split(".")[-1]
    media_type = "image/jpeg" if ext in ("jpg", "jpeg") else f"image/{ext}"
    return base64.standard_b64encode(data).decode("utf-8"), media_type


def analyze_one_image(path, student_text):
    img_b64, media_type = encode_image(path)
    prompt = build_prompt(student_text)
    for attempt in range(MAX_RETRIES):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=350,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": media_type,
                                    "data": img_b64,
                                },
                            },
                            {"type": "text", "text": prompt},
                        ],
                    }
                ],
            )
            text = response.content[0].text.strip()
            text = text.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(text)
            return parsed, text
        except json.JSONDecodeError:
            if attempt == MAX_RETRIES - 1:
                return None, text
        except Exception as e:
            print(f"    retry {attempt+1}/{MAX_RETRIES} after error: {e}")
            time.sleep(2 * (attempt + 1))
    return None, ""


def load_done_ids():
    """Resume support: read which student_ids are already in the output file."""
    if not os.path.exists(OUTPUT_CSV):
        return set()
    done = set()
    with open(OUTPUT_CSV, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            done.add(str(row["student_id"]))
    return done

print("Helper functions defined.")

In [ ]:
def resolve_image_path(value):
    if pd.isna(value) or str(value).strip().lower() in {"", "na", "nan"}:
        return None
    path = Path(value)
    return str((path if path.is_absolute() else REPO_ROOT / path).resolve())

print(f"Using dataset: {DATA_FILE}")
if USING_EXAMPLE_DATA:
    print("Using synthetic example data; labels are for demonstration only.")
print(f"Labels will be saved to: {OUTPUT_CSV}")

In [ ]:
df_test = pd.read_excel(DATA_FILE)
required_columns = {"student_id", IMAGE_COLUMN, "text"}
missing_columns = sorted(required_columns - set(df_test.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")
df_test[IMAGE_COLUMN] = df_test[IMAGE_COLUMN].map(resolve_image_path)

test_candidates = df_test[
    df_test[IMAGE_COLUMN].notna() & (df_test[IMAGE_COLUMN] != "na") &
    df_test["text"].notna() & (df_test["text"] != "na")
]
test_row = test_candidates.iloc[0]
test_path = test_row[IMAGE_COLUMN]
test_text = test_row["text"]
print("Testing on:", test_path)
print("Student text:", test_text)

if not os.path.exists(test_path):
    print("WARNING: file not found at this path. Check your Dataset/ folder structure.")
else:
    parsed, raw = analyze_one_image(test_path, test_text)
    print("Result:", json.dumps(parsed, indent=2, ensure_ascii=False))

## 5. Quick test on ONE image first (recommended)
Before running the full batch, test on a single row to make sure your API key, paths, and the image+text prompt all work.

## 6. Run the full batch (all available image paths)
This can take a while. Progress is saved after every image — safe to interrupt and re-run.

In [ ]:
df = pd.read_excel(DATA_FILE)
print(f"Loaded {len(df)} rows from {DATA_FILE}")
missing_columns = sorted({"student_id", IMAGE_COLUMN, "text"} - set(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")
df[IMAGE_COLUMN] = df[IMAGE_COLUMN].map(resolve_image_path)
done_ids = load_done_ids()
print(f"Already processed: {len(done_ids)} images (will skip these)")

fieldnames = [
    "student_id", "depicted_imagery", "imagery_intensity_vs_text",
    "imagery_specificity", "depicted_body_parts", "raw_response",
]
file_exists = os.path.exists(OUTPUT_CSV)
out_f = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(out_f, fieldnames=fieldnames)
if not file_exists:
    writer.writeheader()
    out_f.flush()

total = len(df)
processed_this_run = 0
failed = []

for idx, row in df.iterrows():
    sid = str(row["student_id"])
    if sid in done_ids:
        continue

    img_path = row.get(IMAGE_COLUMN, None)
    if pd.isna(img_path) or img_path == "na" or not img_path:
        print(f"[{idx+1}/{total}] student_id={sid}: no image path, skipping")
        continue

    if not os.path.exists(img_path):
        print(f"[{idx+1}/{total}] student_id={sid}: FILE NOT FOUND at '{img_path}', skipping")
        failed.append(sid)
        continue

    student_text = row.get("text", "na")
    if pd.isna(student_text):
        student_text = "na"

    print(f"[{idx+1}/{total}] student_id={sid}: analyzing {img_path} ...", end=" ")
    parsed, raw = analyze_one_image(img_path, student_text)

    if parsed is None:
        print("FAILED to parse response")
        failed.append(sid)
        writer.writerow({
            "student_id": sid, "depicted_imagery": "", "imagery_intensity_vs_text": "",
            "imagery_specificity": "", "depicted_body_parts": "", "raw_response": raw,
        })
    else:
        print(f"OK -> {parsed.get('depicted_imagery', '?')} | {parsed.get('imagery_intensity_vs_text', '?')}")
        writer.writerow({
            "student_id": sid,
            "depicted_imagery": parsed.get("depicted_imagery", ""),
            "imagery_intensity_vs_text": parsed.get("imagery_intensity_vs_text", ""),
            "imagery_specificity": parsed.get("imagery_specificity", ""),
            "depicted_body_parts": parsed.get("depicted_body_parts", ""),
            "raw_response": raw,
        })

    out_f.flush()
    processed_this_run += 1
    time.sleep(SLEEP_BETWEEN_CALLS)

out_f.close()

print()
print("=" * 50)
print(f"DONE. Processed {processed_this_run} new images this run.")
print(f"Failed/missing: {len(failed)}")
if failed:
    print(f"Failed student_ids: {failed[:20]}{'...' if len(failed) > 20 else ''}")
print(f"Results saved to: {OUTPUT_CSV}")
print("If the run stopped early or crashed, just re-run this cell - it will resume automatically.")

## 7. Quick check of the output

In [ ]:
result_check = pd.read_csv(OUTPUT_CSV)
print("Total rows in image_labels.csv:", len(result_check))
print()
print("imagery_intensity_vs_text distribution:")
print(result_check["imagery_intensity_vs_text"].value_counts())
print()
print("imagery_specificity distribution:")
print(result_check["imagery_specificity"].value_counts())
result_check.head(10)